> **교육 목표** 시계열을 추세·계절·불규칙 성분으로 분해해 전력 사용량이 어떤 패턴으로 이루어졌는지 파악합니다.
>
> **핵심 내용** 일 단위 분해(주기 7) → 계절 성분 값 → 시간 단위 분해(주기 24) → STL 분해와 비교

In [1]:
%%html
<style>
table {
    float: left;
    clear: both;
}
</style><!-- Markdown 왼쪽 정렬 -->

# D5_02 시계열 분해 (추세 · 계절 · 불규칙)

> **오늘 질문:** 전력 사용량은 어떤 성분이 겹쳐서 만들어졌을까?

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비 📋

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
plt.rcParams["axes.unicode_minus"] = False
from statsmodels.tsa.seasonal import seasonal_decompose, STL

# 1교시에서 한 시간 정리 (00:00 기록 바로잡기 → 구간 시작 시각으로)
se = pd.read_csv("../../data/steel_energy.csv")
se["date"] = pd.to_datetime(se["date"], format="%d/%m/%Y %H:%M")
midnight = (se["date"].dt.hour == 0) & (se["date"].dt.minute == 0)
se.loc[midnight, "date"] += pd.Timedelta(days=1)
se["date"] -= pd.Timedelta(minutes=15)
se = se.set_index("date").sort_index()
daily = se["usage_kwh"].resample("D").sum()
hourly = se["usage_kwh"].resample("h").sum()

## 1. 일 단위 분해 (주기 7) ✍️

`seasonal_decompose`는 이동평균으로 추세를 뽑고, 같은 요일끼리 평균 내 계절을 뽑습니다. 결과 그래프는 위에서부터 원본·추세·계절·불규칙(잔차)입니다.

> 변수 이름: `result`

In [ ]:
# 힌트: seasonal_decompose(시리즈, model="additive", period=7)

## 2. 계절 성분 숫자로 보기 ✍️

일주일치 계절 성분만 꺼내 봅니다. '평균 대비 그 요일에 얼마나 더/덜 쓰나'입니다.

In [ ]:
# 힌트: result.seasonal 은 계절 성분

## 3. 시간 단위 분해 (주기 24) ✍️

시간 단위 데이터로 하루 주기를 봅니다. 그래프가 빽빽하지 않도록 3월 한 달만 분해합니다.

In [ ]:
# 힌트: 시리즈.loc["2018-03"], period=24

## 4. STL 분해와 비교 📋

STL은 계절 모양이 조금씩 변하는 것을 허용하고, 이상치에 덜 흔들립니다.

In [ ]:
stl = STL(daily, period=7, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(10, 7)
plt.show()

compare = pd.DataFrame({"seasonal_decompose 잔차": result.resid, "STL 잔차": stl.resid})
compare.abs().mean().round(0)

## 정리 💬

**Q1.** 일 단위 계절 성분에서 가장 크게 줄어드는 요일은? 주중 대비 얼마나 줄어드나요?

> 답:


**Q2.** 추세 성분과 불규칙 성분에서 눈에 띄는 기간이 있나요? 가법 모형이 적절해 보이나요?

> 답:
